# Exploratory Data Analysis of Climate Models and Precipitation Forecasts

In [1]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [73]:
# Import necessary packages

# pip install if needed
# !pip install hvplot
# !pip install jupyter_bokeh
# !pip install geoviews
import xarray as xr
import hvplot.xarray  # Provides .hvplot methods for xarray objects
import panel as pn
import geoviews as gv   # For geographic overlays (e.g. coastlines)
from cartopy import crs
import holoviews as hv
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Activate the Bokeh backend for interactive plotting
hv.extension('bokeh')
gv.extension('bokeh')
pn.extension()

In [72]:
# # take a model path, combine all years into one year netcdf file
# # For example, all 1991s are combined into 1991_combined

# model_path = '/content/drive/My Drive/capstone_data/NMME/CanESM5/prec/'

# # define list of years
# years = [str(year) for year in range(1981, 2025)]

# # loop through all files, combine them, make sure to change destination paths as desired
# # make sure that the folder is there on google drive
# for year in years:
#   try:

#     model_files = glob.glob(os.path.join(model_path, f'*{year}*.nc'))

#     # open all curret years combined for current model
#     current_year = xr.open_mfdataset(model_files, parallel=True)

#     # save combined nc file to google drive
#     current_year.to_netcdf(f'/content/drive/My Drive/capstone_data/NMME/CanESM5/prec_combined_years/prec.CanESM5.{year}_combined.nc')

#     print(f"Succesfully Combined {year}")
#   except:
#     print(f'Error with {year}')

# print(gfdl_files)

# Interactive Graph for one year at a time GFDL

In [51]:
# start with gfdl, one year

# year is changeable
years = ['1991', '1992', '1993']

# open the combined nc path for one year
gfdl_year_path = f'/content/drive/My Drive/capstone_data/NMME/GFDL-SPEAR/prec_combined_years/prec.GFDL-SPEAR.{years[0]}_combined.nc'

# open combined year dataset
GFDL = xr.open_dataset(gfdl_year_path)

# Rename
GFDL = GFDL.rename({'Y': 'latitude',
                    'X': 'longitude',
                    'prec': 'precip',
                    'S': 'time',
                    'L': 'lead_time',
                    'M': 'ensemble_member'})

In [69]:
# Ensure the "time" coordinate is in datetime format
GFDL["time"] = xr.decode_cf(GFDL)["time"]

# Extract unique values for the widgets
year = GFDL["time"].dt.year.values[0]
lead_times = sorted(set(GFDL["lead_time"].values))
ensemble_members = sorted(set(GFDL["ensemble_member"].values))

# Create Panel widgets
month_slider = pn.widgets.IntSlider(name="Month", start=min(months), end=max(months), step=1, value=5)
lead_slider = pn.widgets.FloatSlider(name="Lead Time", start=0.5, end=11.5, step=1, value=0.5)
ensemble_selector = pn.widgets.Select(name="Ensemble Member", options=ensemble_members, value=ensemble_members[0])

# Define interactive function for predictive model
def plot_precip_heatmap(lead_time, model):
    filtered_ds = model.sel(lead_time=lead_time)

    # the parameters in Orthographic() projection function control the rotation, central_latitude=0, central_longitude=0
    # Other projections: PlateCarree, Robinson, LambertCylindrical, Mercator, Miller, Mollweide, InterruptedGoodeHomolosine
    # Other cmap colors: Spectral, RdYlBu, coolwarm
    return filtered_ds["precip"].hvplot.quadmesh(
    x="longitude", y="latitude", colorbar=True,
    projection=crs.PlateCarree(), project=True,
    global_extent=True, cmap='RdYlBu', coastline=True,
    title=f"Precipitation forecasts for {year} - Lead: {lead_time}"
)

# Bind widgets with function
interactive_plot = pn.bind(plot_precip_heatmap, lead_time=lead_slider, model=GFDL)

# Display
pn.Column(lead_slider, interactive_plot).servable()

Column
    [0] FloatSlider(end=11.5, name='Lead Time', start=0.5, step=1, value=0.5)
    [1] ParamFunction(function, _pane=Row, defer_load=False)

# View CHIRPS Interactively (WIP)

In [ ]:
# # Running an interactive map for chirps is super slow and memory intensive, look into subsetting
# # Create a dictionary with regions of interest, make it into a slider, have it subset with the slider

# # Load in chirps
# chirps_path = '/content/drive/My Drive/capstone_data/CHIRPS/*.nc'

# chirps = xr.open_mfdataset(chirps_path, parallel=True)

# # regional dict
# regional_dict = {
#     'South_Sudan': {'longitude': (3, 13), 'latitude': (25, 35)},
#      'Kenya': {'longitude': (34, 42), 'latitude': (-5, 5)}
# }

In [ ]:
# # Ensure the "time" coordinate is in datetime format (if not already)
# chirps["time"] = xr.decode_cf(chirps)["time"]

# # Extract unique values for the widgets
# months = sorted(set(chirps["time"].dt.month.values))
# years = sorted(set(chirps["time"].dt.year.values))

# # Create interactive widgets
# year_slider = pn.widgets.IntSlider(name="Year", start=1981, end=2025, step=1, value=2020)
# month_slider = pn.widgets.IntSlider(name="Month", start=1, end=12, step=1, value=6)

# # # Runs out of ram, replace region with region_selector if more ram
# # region_selector = pn.widgets.Select(name="Region", options=list(regional_dict.keys()))

In [ ]:
# # Define interactive function for chirps
# def plot_precip_heatmap_chirps(month, year, region_dict, region):

#   min_lat = region_dict[region]['latitude'][0]
#   max_lat = region_dict[region]['latitude'][1]
#   min_lon = region_dict[region]['longitude'][0]
#   max_lon = region_dict[region]['longitude'][1]

#   filtered_ds = chirps.where(
#     (chirps["time"].dt.month == month) & (chirps["time"].dt.year == year) &
#     (chirps["longitude"] >= min_lon) & (chirps["longitude"] <= max_lon) &
#     (chirps["latitude"] >= min_lat) & (chirps["latitude"] <= max_lat),
#     drop=True
# )

#     # the parameters in Orthographic() function control the rotation
#     # for some reason, projections like Robinson dont work, it bugs out
#   return filtered_ds["precip"].hvplot.quadmesh(
#   x="longitude", y="latitude", colorbar=True,
#   projection=crs.Orthographic(central_latitude=0, central_longitude=0), project=True,
#   global_extent=True, cmap='jet', coastline=True,
#   title=f"Chirps Precipitation - Month: {month}, Year: {year}" # lead_time removed, was not defined
# )

In [ ]:
# # Link sliders to function
# interactive_map = pn.bind(plot_precip_heatmap_chirps, month=month_slider, year=year_slider, region_dict=regional_dict, region='South_Sudan')

# # Display
# pn.Column(month_slider, year_slider, interactive_map).servable()